# Raw Data Download

Download raw experimental data files from the aimdl/datafiles API with configurable data types.

This notebook demonstrates how to:
- Query files by data type using the aimdl/datafiles endpoint
- Download multiple files in parallel
- Save to disk with metadata

In [1]:
import io
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from girder_client import GirderClient

/Users/alirachidi/miniconda3/lib/python3.9/site-packages/girder_client/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


## Setup

In [2]:
def get_girder_client(session=None):
    gc = GirderClient(
        apiUrl=os.environ.get("GIRDER_API_URL", "https://girder.htmdec.org/api/v1")
    )
    gc._session = session
    try:
        gc.authenticate(apiKey=os.environ["GIRDER_API_KEY"])
    except KeyError:
        raise RuntimeError("GIRDER_API_KEY environment variable not set")

    me = gc.get("user/me")
    assert me is not None, "Failed to authenticate with Girder API"
    return gc


def download_item_to_disk(gc, item, output_dir):
    """Download a Girder item to disk."""
    item_id = item["_id"]
    filename = item["name"]
    filepath = Path(output_dir) / filename
    filepath.parent.mkdir(parents=True, exist_ok=True)
    
    response = gc.sendRestRequest(
        "GET", f"item/{item_id}/download", stream=True, jsonResp=False
    )
    response.raise_for_status()
    
    with open(filepath, 'wb') as f:
        for chunk in response.iter_content(chunk_size=65536):
            f.write(chunk)
    
    return {"name": filename, "path": str(filepath), "size": filepath.stat().st_size, "itemId": item_id}

## Configuration

In [ ]:
from dotenv import load_dotenv

load_dotenv('..')

# Configure data type to download
# Change this to fetch different data types (e.g., "pdv_raw", "xrd_raw", etc.)
DATA_TYPE = "pdv_raw"

# Output directory for downloaded files
OUTPUT_DIR = f"./raw_data_{DATA_TYPE}"

## Download Files

In [4]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    results = []
    limit = 50
    offset = 0
    
    with ThreadPoolExecutor(max_workers=8) as executor:
        while True:
            batch = gc.get(
                "aimdl/datafiles",
                parameters={
                    "limit": limit,
                    "offset": offset,
                    "dataType": DATA_TYPE,
                },
            )
            print(batch)
            break
            if not batch:
                break
            
            futures = [
                executor.submit(download_item_to_disk, gc, item, OUTPUT_DIR)
                for item in batch
            ]
            
            for future in as_completed(futures):
                try:
                    result = future.result()
                    results.append(result)
                    print(f"Downloaded: {result['name']} ({result['size'] / 1024:.2f} KB)")
                except Exception as e:
                    print(f"Error: {e}")
            
            if len(batch) < limit:
                break
            offset += limit

print(f"\\nTotal files downloaded: {len(results)}")
print(f"Total size: {sum(r['size'] for r in results) / (1024**2):.2f} MB")

[]
\nTotal files downloaded: 0
Total size: 0.00 MB


## Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(f"Downloaded {len(df)} files of type '{DATA_TYPE}'")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\\nSample files:")
print(df[["name", "size"]].head(10))